In [33]:
def replace_all(text, dic):
    for i, j in dic.items():
        if j != 'nan':
            text = text.replace(i, j)
    return text


# def check_if_filename_is_dir(file, i):
#     if os.path.isdir(i.replace('$PLANT_NAME', file)):
#         return file
#     else:
#         return 'nan'

# needs to be included in processing script


def generate_makeflow_json(cctools_path,level, files_list, command, container, inputs, outputs, date, sensor, n_rules=1, json_out_path='wf_file.json'):
    i = 'nan'

    if level == 'subdir':
        subdir_list = []
        for item in files_list:
            subdir = os.path.basename(os.path.dirname(item))
            subdir_list.append(subdir)
        files_list = list(set(subdir_list))

        write_file_list(files_list)


    jx_dict = {
                        "rules": []
    }

    rules = []

    for file in files_list:

        variable_dict = {
            'command_variables': {
            # s10 3D
            '${FILE}': file,
            '${SEG_MODEL_PATH}': seg_model_name,
            '${DET_MODEL_PATH}': det_model_name,
            # '${PLANT_NAME}': file, # conflicting name plant_name change to file

            # s11 3d
            '${FILE}': file,
            '${PLANT_PATH}': os.path.dirname(file), 
            '${SEG_MODEL_PATH}': seg_model_name,
            '${PLANT_NAME}': os.path.basename(os.path.dirname(file)),
            '${DET_MODEL_PATH}': det_model_name,
            '${SUBDIR}': os.path.basename(os.path.dirname(file)),
            '${DATE}': date,
            '${INPUT_DIR}': os.path.dirname(file),


            #s10 ps2
            '${UUID}': os.path.basename(file).split('_')[0],
            '${FILE}': file,
            '${S_D_PATH}': os.path.dirname(file), #conflicting name sub_dir

            # not specified
            '${FILE}': file,
            '${PLANT_PATH}': os.path.dirname(file),
            '${SEG_MODEL_PATH}': seg_model_name,
            '${PLANT_NAME}': os.path.basename(os.path.dirname(file)),
            '${DET_MODEL_PATH}': det_model_name,
            '${SUBDIR}': os.path.basename(os.path.dirname(file)),
            '${DATE}': date,

            # no inputs
            '${FILE}': file,
            '${PLANT_PATH}': os.path.dirname(file),
            '${SEG_MODEL_PATH}': seg_model_name,
            '${PLANT_NAME}': os.path.basename(os.path.dirname(file)),
            '${DET_MODEL_PATH}': det_model_name,
            '${SUBDIR}': os.path.basename(os.path.dirname(file)),
            '${DATE}': date,

            
            },

        

            'output_variables':{
                #s10 3d
                # '$PLANT_NAME': file, # conflicting name plant_name, change to file

                #s11 3d
                '$UUID': '_'.join(os.path.basename(file).split('_')[:2]),
                '$PLANT_NAME': os.path.basename(os.path.dirname(file)),
                '$SUBDIR': os.path.join(os.path.basename(os.path.dirname(file)), os.path.basename(file)),
                '${DATE}': date,
                '$BASENAME': os.path.basename(os.path.dirname(file)),

                #s10 ps2
                '$FILE_BASE': os.path.basename(file),
                '.bin': '',
                #not specified
                '$PLANT_NAME': os.path.basename(os.path.dirname(file)),
                '$SUBDIR': os.path.join(os.path.basename(os.path.dirname(file)), os.path.basename(file)),
                '${DATE}': date,

                # no inputs
                '$PLANT_NAME': os.path.basename(os.path.dirname(file)),
                '$SUBDIR': os.path.join(os.path.basename(os.path.dirname(file)), os.path.basename(file)),
                '${DATE}': date,

            },



            'input_variables':{
                #s10 3d
                # '$PLANT_NAME': check_if_filename_is_dir(file, i), Needs to be included in processing script

                #s11 3d
                '$PLANT_NAME': os.path.basename(os.path.dirname(file)),
                '$SUBDIR': os.path.join(os.path.basename(os.path.dirname(file)), os.path.basename(file)),
                '${DATE}': date,
                '$FILE': file,

                #s10 ps2
                '$S_D_PATH': os.path.dirname(file),
                '$UUID':os.path.basename(file).split('_')[0],
                '$FILE': file,

                #not specifieds
                '$PLANT_NAME': os.path.basename(os.path.dirname(file)),
                '$SUBDIR': os.path.join(os.path.basename(os.path.dirname(file)), os.path.basename(file)),
                '${DATE}': date,
                
                # no inputs

            }
        }

        command = replace_all(command, variable_dict['command_variables'])

        output_list = [replace_all(i, variable_dict['output_variables']) for i in outputs]

        input_list = [replace_all(i, variable_dict['input_variables']) for i in inputs]

        a_rule = {
            "command" : timeout + command,
            "outputs" : output_list,
            "inputs"  : [container, 
                        seg_model_name, 
                        det_model_name] + input_list
        }

        rules.append(a_rule)
        break

    jx_dict['rules'] = rules

    print(jx_dict)

In [35]:
import json
import glob
import yaml

files_list = glob.glob('/home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/*')
yaml_path = '/home/travis/repos/psii_testing_may_4/parser_change/automation/yaml_files/season_10/s10_psii.yaml'
cctools_path = 'cctools_path'
date = '2022-20-20'

det_model_name =  '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/detecto_heatmap_lettuce_detection_weights.pth'
seg_model_name =  '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/dgcnn_3d_model.pth'
timeout = '1 hour '

with open(yaml_path, 'r') as stream:
        try:
            global dictionary
            dictionary = yaml.safe_load(stream)
            # build_containers(dictionary)

        except yaml.YAMLError as exc:
            print(exc)

        for k, v in dictionary['modules'].items():
            generate_makeflow_json(cctools_path=cctools_path, level=v['file_level'], files_list=files_list, command=v['command'], container=v['container']['simg_name'], inputs=v['inputs'], outputs=v['outputs'], date=date, sensor=dictionary['tags']['sensor'], json_out_path=f'wf_file_{k}.json')




{'rules': [{'command': '1 hour singularity run psii_bin_to_tif.simg /home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043.bin -m /home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_metadata.json', 'outputs': ['bin2tif_out/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043.tif'], 'inputs': ['psii_bin_to_tif.simg', '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/dgcnn_3d_model.pth', '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/detecto_heatmap_lettuce_detection_weights.pth', '/home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043.bin', '/home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_metadata.json']}]}
{'rules': [{'command': '1 hour sing

In [36]:
import json
import glob
import yaml

files_list = glob.glob('/home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/*')
yaml_path = '/home/travis/repos/psii_testing_may_4/parser_change/automation/yaml_files/season_11/3d_pre_landmark_selection.yaml'
cctools_path = 'cctools_path'
date = '2022-20-20'

det_model_name =  '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/detecto_heatmap_lettuce_detection_weights.pth'
seg_model_name =  '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/dgcnn_3d_model.pth'
timeout = '1 hour '

with open(yaml_path, 'r') as stream:
        try:
            global dictionary
            dictionary = yaml.safe_load(stream)
            # build_containers(dictionary)

        except yaml.YAMLError as exc:
            print(exc)

        for k, v in dictionary['modules'].items():
            generate_makeflow_json(cctools_path=cctools_path, level=v['file_level'], files_list=files_list, command=v['command'], container=v['container']['simg_name'], inputs=v['inputs'], outputs=v['outputs'], date=date, sensor=dictionary['tags']['sensor'], json_out_path=f'wf_file_{k}.json')



{'rules': [{'command': '1 hour singularity run 3d_preprocessing.simg -l gcp_season_11.txt -o alignment -i /home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160', 'outputs': ['alignment/east/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-east.ply', 'alignment/east_downsampled/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-east.ply', 'alignment/merged/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-merged.ply', 'alignment/merged_downsampled/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-merged.ply', 'alignment/west/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-west.ply', 'alignment/west_downsampled/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-west.ply', 'alignment/metadata/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39

In [37]:
import json
import glob
import yaml

files_list = glob.glob('/home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160/*')
yaml_path = '/home/travis/repos/psii_testing_may_4/parser_change/automation/yaml_files/season_11/3d_post_landmark_selection.yaml'
cctools_path = 'cctools_path'
date = '2022-20-20'

det_model_name =  '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/detecto_heatmap_lettuce_detection_weights.pth'
seg_model_name =  '/iplant/home/shared/phytooracle/season_10_lettuce_yr_2020/level_0/necessary_files/dgcnn_3d_model.pth'
timeout = '1 hour '

with open(yaml_path, 'r') as stream:
        try:
            global dictionary
            dictionary = yaml.safe_load(stream)
            # build_containers(dictionary)

        except yaml.YAMLError as exc:
            print(exc)

        for k, v in dictionary['modules'].items():
            generate_makeflow_json(cctools_path=cctools_path, level=v['file_level'], files_list=files_list, command=v['command'], container=v['container']['simg_name'], inputs=v['inputs'], outputs=v['outputs'], date=date, sensor=dictionary['tags']['sensor'], json_out_path=f'wf_file_{k}.json')


mapping values are not allowed here
  in "/home/travis/repos/psii_testing_may_4/parser_change/automation/yaml_files/season_11/3d_post_landmark_selection.yaml", line 181, column 22
{'rules': [{'command': '1 hour singularity run 3d_preprocessing.simg -l gcp_season_11.txt -o alignment -i /home/travis/repos/psii_testing_may_4/ps2Top/2020-02-26/2020-02-26__21-44-30-160', 'outputs': ['alignment/east/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-east.ply', 'alignment/east_downsampled/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-east.ply', 'alignment/merged/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-merged.ply', 'alignment/merged_downsampled/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-merged.ply', 'alignment/west/2020-02-26__21-44-30-160/1d1c248e-946a-4de5-bae3-39396c3e2ade_rawData0043__Top-heading-west.ply', 'alignment/west_d